# Can AI Know When It Is Wrong?
## Measuring predictive-confidence calibration in FLAN-T5-base

**Stanford Probability for AI · Final project**  
**Model:** `google/flan-t5-base`  
**Dataset:** MMLU (`cais/mmlu`)  
**Domains:** high-school biology, high-school computer science, high-school statistics

### What changed in this corrected notebook?
The first version asked FLAN-T5 to *say* a confidence percentage. The saved outputs showed that many confidence responses were not valid percentages (for example, answer letters or missing values), so those calibration results were not dependable.

This version removes that fragile self-report step. For each question, it scores all four answer choices (A–D) using the model's conditional sequence likelihood, converts those four scores into a probability distribution with softmax, and uses the largest probability as predictive confidence. The predicted answer is the option with the largest score.

This is **model-derived predictive confidence**, not a self-reported percentage and not proof that the probabilities are calibrated. The project therefore tests whether the model's option probabilities align with observed accuracy.

> **Important research-integrity note:** the earlier test results were already viewed. This corrected run is a revised exploratory analysis, not a pristine blind replication. Do not describe its test split as never-before-seen across the entire project. The split is held out from development *within this corrected run*.

### Before running
Write your own prediction before executing the experiment. Edit the statement below if it does not honestly represent your prior belief. Do not change it after seeing results.

In [ ]:
# PRE-REGISTER YOUR PREDICTION BEFORE RUNNING THE EXPERIMENT.
PREDICTION = (
    "I expect FLAN-T5-base's predictive confidence to be imperfectly calibrated, "
    "with calibration differing across the three subjects."
)
print("Pre-experiment prediction:", PREDICTION)

## 1. Install dependencies
Run in Kaggle or Google Colab. A GPU is helpful but not required. On Kaggle, turn Internet on in Notebook settings so the public Hugging Face model and dataset can download. A Hugging Face token is optional for these public assets.

In [ ]:
!pip -q install -U datasets transformers sentencepiece accelerate pandas matplotlib scikit-learn tqdm

## 2. Load MMLU and create development / held-out splits
We sample up to 100 questions per subject and split each subject into 70% development and 30% held-out. The seed is deliberately different from the earlier notebook. Because some subject test sets are small and the earlier run was inspected, this does not erase the earlier exposure; see the research-integrity note above.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import random, json, os, math, gc
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 20261118
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SUBJECTS = {
    "Biology": "high_school_biology",
    "Computer science": "high_school_computer_science",
    "Statistics": "high_school_statistics",
}

all_rows = []
for domain, subject in SUBJECTS.items():
    ds = load_dataset("cais/mmlu", subject, split="test")
    frame = ds.to_pandas()
    frame = frame.sample(n=min(100, len(frame)), random_state=SEED).reset_index(drop=True)
    frame["domain"] = domain
    frame["source_subject"] = subject
    frame["question_id"] = [f"{subject}_{SEED}_{i:03d}" for i in range(len(frame))]
    all_rows.append(frame)

all_data = pd.concat(all_rows, ignore_index=True)
dev_parts, test_parts = [], []
for domain, group in all_data.groupby("domain", sort=True):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_test = max(1, round(len(group) * 0.30))
    test_parts.append(group.iloc[:n_test].copy())
    dev_parts.append(group.iloc[n_test:].copy())

dev_data = pd.concat(dev_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
heldout_data = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Development questions:", len(dev_data))
print("Held-out questions:", len(heldout_data))
print("Development by subject:\n", dev_data.domain.value_counts())
print("Held-out by subject:\n", heldout_data.domain.value_counts())

## 3. Load the model
We use one fixed model. The model is evaluated in inference mode; no fine-tuning is performed.

In [ ]:
MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"Loaded {MODEL_NAME} on {DEVICE}")

## 4. Estimate probabilities over the four answer choices
For each question, the model scores the likelihood of each candidate answer letter conditioned on the same question prompt. We apply softmax to the four scores. The resulting values sum to one and represent the model's relative predictive distribution over the provided options.

This is more reproducible than asking the model to invent a percentage, but it is still not automatically a calibrated probability. The score depends on the model, tokenizer, prompt, and candidate set.

In [ ]:
LETTERS = ["A", "B", "C", "D"]

def make_prompt(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{letter}. {choice}" for letter, choice in zip(LETTERS, choices))
    return (
        "Answer the following multiple-choice question. Select the best option.\n"
        f"Question: {row['question']}\n{choices_text}\nAnswer:"
    )

def option_log_scores(prompt):
    # Repeat the same encoded question once for each possible answer label.
    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    input_ids = encoded["input_ids"].repeat(4, 1).to(DEVICE)
    attention_mask = encoded["attention_mask"].repeat(4, 1).to(DEVICE)

    target_enc = tokenizer(
        text_target=LETTERS,
        return_tensors="pt",
        padding=True,
        add_special_tokens=True
    )
    labels = target_enc["input_ids"].to(DEVICE)
    # Ignore padding in the sequence likelihood.
    labels = labels.masked_fill(labels == tokenizer.pad_token_id, -100)
    # All four answer labels must have equal tokenized length for comparable summed log-likelihoods.
    lengths = (labels != -100).sum(dim=1)
    if not torch.all(lengths == lengths[0]):
        raise ValueError(f"A-D labels have unequal token lengths: {lengths.tolist()}")

    with torch.no_grad():
        # HF seq2seq loss is mean token cross-entropy per candidate; convert to summed log-likelihood.
        individual_scores = []
        for i in range(4):
            out = model(
                input_ids=input_ids[i:i+1],
                attention_mask=attention_mask[i:i+1],
                labels=labels[i:i+1]
            )
            individual_scores.append(-out.loss * lengths[i].to(out.loss.dtype))
        scores = torch.stack(individual_scores)
        probs = torch.softmax(scores, dim=0)
    return scores.detach().cpu().numpy(), probs.detach().cpu().numpy()

def predict_row(row):
    prompt = make_prompt(row)
    scores, probs = option_log_scores(prompt)
    pred_idx = int(np.argmax(probs))
    correct_idx = int(row["answer"])
    return {
        "question_id": row["question_id"],
        "domain": row["domain"],
        "model_answer": LETTERS[pred_idx],
        "correct_answer": LETTERS[correct_idx],
        "correct": int(pred_idx == correct_idx),
        "confidence": float(probs[pred_idx]),
        "p_A": float(probs[0]), "p_B": float(probs[1]),
        "p_C": float(probs[2]), "p_D": float(probs[3]),
        "score_A": float(scores[0]), "score_B": float(scores[1]),
        "score_C": float(scores[2]), "score_D": float(scores[3]),
    }

# Quick sanity check: four probabilities should sum to approximately one.
_demo = predict_row(dev_data.iloc[0])
print("Sanity-check probabilities:", [_demo[f"p_{x}"] for x in LETTERS])
print("Sum:", sum(_demo[f"p_{x}"] for x in LETTERS))

## 5. Run development inference
The output CSV is cached so rerunning later cells does not repeat inference. If you change the model, prompt, or seed, change `RUN_TAG` so you do not accidentally reuse an old CSV.

In [ ]:
RUN_TAG = "flan_t5_option_probs_seed20261118_v1"

def run_split(frame, output_path):
    if os.path.exists(output_path):
        print(f"Loading cached predictions: {output_path}")
        cached = pd.read_csv(output_path)
        if len(cached) == len(frame) and set(cached.question_id) == set(frame.question_id):
            return cached
        print("Cache does not match this split; recomputing.")
    records = []
    for _, row in tqdm(frame.iterrows(), total=len(frame)):
        records.append(predict_row(row))
    result = pd.DataFrame(records)
    result.to_csv(output_path, index=False)
    return result

dev_results = run_split(dev_data, f"development_{RUN_TAG}.csv")
print("Development accuracy:", round(dev_results.correct.mean() * 100, 2), "%")
print("Development rows:", len(dev_results))

## 6. Calibration analysis
We use five fixed confidence bins. For each bin, compare mean predicted confidence with empirical accuracy. The absolute calibration gap is their absolute difference. The expected calibration error (ECE) is the bin gap weighted by the number of observations in each bin. This estimate depends on binning and sample size.

In [ ]:
BIN_EDGES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0000001]
BIN_LABELS = ["0–19%", "20–39%", "40–59%", "60–79%", "80–100%"]

def calibration_table(results):
    valid = results.dropna(subset=["confidence", "correct"]).copy()
    valid["confidence_bin"] = pd.cut(
        valid.confidence, bins=BIN_EDGES, labels=BIN_LABELS,
        right=False, include_lowest=True
    )
    rows = []
    for label in BIN_LABELS:
        part = valid[valid.confidence_bin == label]
        if len(part):
            mean_conf = float(part.confidence.mean())
            accuracy = float(part.correct.mean())
            rows.append({
                "confidence_bin": label, "n": int(len(part)),
                "mean_confidence": mean_conf, "accuracy": accuracy,
                "absolute_gap": abs(mean_conf - accuracy),
                "direction": "overconfident" if mean_conf > accuracy else "underconfident" if mean_conf < accuracy else "matched"
            })
    return pd.DataFrame(rows)

def expected_calibration_error(table):
    if table.empty: return float("nan")
    return float((table.absolute_gap * table.n).sum() / table.n.sum())

def show_summary(label, results):
    table = calibration_table(results)
    print(f"{label} questions: {len(results)}")
    print(f"{label} accuracy: {results.correct.mean()*100:.2f}%")
    print(f"{label} ECE: {expected_calibration_error(table)*100:.2f} percentage points")
    display(table)
    return table

dev_table = show_summary("Development", dev_results)

In [ ]:
def plot_reliability(results, title, filename):
    table = calibration_table(results)
    plt.figure(figsize=(7, 6))
    plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
    if not table.empty:
        plt.plot(table.mean_confidence, table.accuracy, "o-", linewidth=2, label="FLAN-T5-base")
        for _, r in table.iterrows():
            plt.annotate(f"n={int(r.n)}", (r.mean_confidence, r.accuracy), xytext=(5, 5), textcoords="offset points")
    plt.xlabel("Mean predictive confidence")
    plt.ylabel("Observed fraction correct")
    plt.title(title)
    plt.xlim(0, 1); plt.ylim(0, 1); plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_reliability(dev_results, "Development: predictive confidence vs accuracy", "development_reliability.png")

## 7. Compare subjects (development only)

In [ ]:
subject_rows = []
for domain, part in dev_results.groupby("domain"):
    tab = calibration_table(part)
    subject_rows.append({
        "domain": domain, "n": int(len(part)),
        "accuracy": float(part.correct.mean()),
        "ECE": expected_calibration_error(tab)
    })
subject_summary = pd.DataFrame(subject_rows)
display(subject_summary)

## 8. Evaluate the held-out split
Complete your development interpretation before running this section. In this corrected notebook, the held-out split is held out from development within this run, but remember that the earlier version's test outputs were already inspected. Do not claim this is a fully untouched test across all project iterations.

In [ ]:
heldout_results = run_split(heldout_data, f"heldout_{RUN_TAG}.csv")
heldout_table = show_summary("Held-out", heldout_results)
plot_reliability(heldout_results, "Held-out: predictive confidence vs accuracy", "heldout_reliability.png")

## 9. Export reproducible results

In [ ]:
summary = {
    "project_title": "Can AI Know When It Is Wrong?",
    "model": MODEL_NAME,
    "dataset": "cais/mmlu",
    "confidence_method": "softmax over conditional sequence log-likelihoods for options A-D",
    "seed": SEED,
    "prediction": PREDICTION,
    "development": {
        "n_questions": int(len(dev_results)),
        "accuracy": float(dev_results.correct.mean()),
        "ECE": expected_calibration_error(dev_table),
        "calibration_bins": dev_table.to_dict(orient="records")
    },
    "heldout": {
        "n_questions": int(len(heldout_results)),
        "accuracy": float(heldout_results.correct.mean()),
        "ECE": expected_calibration_error(heldout_table),
        "calibration_bins": heldout_table.to_dict(orient="records")
    },
    "subject_development": subject_summary.to_dict(orient="records"),
    "limitations": [
        "The earlier version's test results were already viewed; this corrected run is exploratory, not a pristine blind replication.",
        "The sample is small and results apply only to this model, prompt, dataset, and selected subjects.",
        "MMLU is public and may have appeared in model pretraining data.",
        "Softmax-normalized option likelihoods are predictive scores, not guaranteed calibrated probabilities.",
        "ECE depends on binning and can be unstable with small samples."
    ]
}
with open("project_results_corrected.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved project_results_corrected.json")
print("Also save these files from the Kaggle output panel:")
for name in [f"development_{RUN_TAG}.csv", f"heldout_{RUN_TAG}.csv", "development_reliability.png", "heldout_reliability.png", "project_results_corrected.json"]:
    print(" -", name)

## 10. Interpretation checklist
In the final write-up and video:

1. State your prediction exactly as registered before running.
2. Report both development and held-out sample sizes, accuracy, and ECE.
3. Include the reliability diagrams and note the number of observations in each bin.
4. Explain that confidence is derived from relative likelihoods across the four available answer choices, not a verbal self-report.
5. Discuss the small sample, possible MMLU contamination, prompt/model dependence, binning sensitivity, and prior exposure of the earlier test results.
6. Do not claim that one small benchmark establishes how AI systems behave generally.

**No performance results are pre-filled.** Use only the values produced by your own run.